# Semana 2: Modulación y Demodulación Digital en Canales AWGN

## Objetivo
Implementar y analizar esquemas de modulación digital (ASK, FSK, BPSK) en canales con ruido gaussiano aditivo blanco (AWGN).

## Ecuaciones de Modulación

### ASK (Amplitude Shift Keying)
$$s(t) = A b(t) \cos(2\pi f_c t)$$

### FSK (Frequency Shift Keying)
$$s(t) = A \cos(2\pi f_i t), \quad f_i \in \{f_0, f_1\}$$

### BPSK (Binary Phase Shift Keying)
$$s(t) = A m(t) \cos(2\pi f_c t), \quad m(t) \in \{-1, +1\}$$

In [ ]:
# Instalación de dependencias
!pip install numpy matplotlib scipy pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import pandas as pd
from typing import Tuple, List

# Configuración de matplotlib
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## Algoritmo de Implementación

### Paso 1: Generar bits aleatorios
### Paso 2: Aplicar modulación según esquema seleccionado
### Paso 3: Pasar señal por canal AWGN
### Paso 4: Demodular señal recibida
### Paso 5: Calcular BER (Bit Error Rate)

In [ ]:
class DigitalModulation:
    def __init__(self, fc: float = 1000, fs: float = 8000, tb: float = 0.001, A: float = 1.0):
        """
        Inicializar parámetros de modulación.
        
        Args:
            fc: Frecuencia portadora (Hz)
            fs: Frecuencia de muestreo (Hz)
            tb: Duración de bit (s)
            A: Amplitud de la señal
        """
        self.fc = fc
        self.fs = fs
        self.tb = tb
        self.A = A
        self.samples_per_bit = int(fs * tb)
        self.t_bit = np.linspace(0, tb, self.samples_per_bit, endpoint=False)
    
    def generate_bits(self, N: int) -> np.ndarray:
        """Generar N bits aleatorios."""
        return np.random.randint(0, 2, N)
    
    def ask_modulate(self, bits: np.ndarray) -> np.ndarray:
        """
        Modulación ASK: s(t) = A * b(t) * cos(2πfct)
        b(t) = 1 para bit '1', b(t) = 0 para bit '0'
        """
        signal_mod = np.array([])
        
        for bit in bits:
            if bit == 1:
                s_bit = self.A * np.cos(2 * np.pi * self.fc * self.t_bit)
            else:
                s_bit = np.zeros_like(self.t_bit)  # Sin portadora para '0'
            
            signal_mod = np.concatenate([signal_mod, s_bit])
        
        return signal_mod
    
    def fsk_modulate(self, bits: np.ndarray, f0: float = None, f1: float = None) -> np.ndarray:
        """
        Modulación FSK: s(t) = A * cos(2πfit)
        fi = f0 para bit '0', fi = f1 para bit '1'
        """
        if f0 is None:
            f0 = self.fc - self.fc * 0.1  # 10% menor que fc
        if f1 is None:
            f1 = self.fc + self.fc * 0.1  # 10% mayor que fc
            
        signal_mod = np.array([])
        
        for bit in bits:
            if bit == 1:
                s_bit = self.A * np.cos(2 * np.pi * f1 * self.t_bit)
            else:
                s_bit = self.A * np.cos(2 * np.pi * f0 * self.t_bit)
            
            signal_mod = np.concatenate([signal_mod, s_bit])
        
        return signal_mod
    
    def bpsk_modulate(self, bits: np.ndarray) -> np.ndarray:
        """
        Modulación BPSK: s(t) = A * m(t) * cos(2πfct)
        m(t) = +1 para bit '1', m(t) = -1 para bit '0'
        """
        signal_mod = np.array([])
        
        for bit in bits:
            if bit == 1:
                m_t = 1
            else:
                m_t = -1
            
            s_bit = self.A * m_t * np.cos(2 * np.pi * self.fc * self.t_bit)
            signal_mod = np.concatenate([signal_mod, s_bit])
        
        return signal_mod
    
    def awgn_channel(self, signal: np.ndarray, snr_db: float) -> np.ndarray:
        """
        Canal AWGN: agregar ruido gaussiano blanco.
        
        Args:
            signal: Señal de entrada
            snr_db: Relación señal-ruido en dB
        
        Returns:
            Señal con ruido
        """
        # Calcular potencia de la señal
        signal_power = np.mean(signal**2)
        
        # Convertir SNR de dB a lineal
        snr_linear = 10**(snr_db / 10)
        
        # Calcular potencia del ruido
        noise_power = signal_power / snr_linear
        
        # Generar ruido gaussiano
        noise = np.sqrt(noise_power) * np.random.randn(len(signal))
        
        return signal + noise
    
    def ask_demodulate(self, received_signal: np.ndarray, N: int) -> np.ndarray:
        """Demodulación ASK usando detector de energía."""
        detected_bits = np.zeros(N, dtype=int)
        
        for i in range(N):
            start_idx = i * self.samples_per_bit
            end_idx = (i + 1) * self.samples_per_bit
            bit_signal = received_signal[start_idx:end_idx]
            
            # Calcular energía del bit
            energy = np.sum(bit_signal**2)
            
            # Umbral de decisión
            threshold = (self.A**2 * self.samples_per_bit) / 4
            
            detected_bits[i] = 1 if energy > threshold else 0
        
        return detected_bits
    
    def fsk_demodulate(self, received_signal: np.ndarray, N: int, f0: float = None, f1: float = None) -> np.ndarray:
        """Demodulación FSK usando filtros correladores."""
        if f0 is None:
            f0 = self.fc - self.fc * 0.1
        if f1 is None:
            f1 = self.fc + self.fc * 0.1
            
        detected_bits = np.zeros(N, dtype=int)
        
        # Señales de referencia
        ref_0 = np.cos(2 * np.pi * f0 * self.t_bit)
        ref_1 = np.cos(2 * np.pi * f1 * self.t_bit)
        
        for i in range(N):
            start_idx = i * self.samples_per_bit
            end_idx = (i + 1) * self.samples_per_bit
            bit_signal = received_signal[start_idx:end_idx]
            
            # Correlación con señales de referencia
            corr_0 = np.sum(bit_signal * ref_0)
            corr_1 = np.sum(bit_signal * ref_1)
            
            detected_bits[i] = 1 if corr_1 > corr_0 else 0
        
        return detected_bits
    
    def bpsk_demodulate(self, received_signal: np.ndarray, N: int) -> np.ndarray:
        """Demodulación BPSK usando detector coherente."""
        detected_bits = np.zeros(N, dtype=int)
        
        # Señal de referencia
        ref_signal = np.cos(2 * np.pi * self.fc * self.t_bit)
        
        for i in range(N):
            start_idx = i * self.samples_per_bit
            end_idx = (i + 1) * self.samples_per_bit
            bit_signal = received_signal[start_idx:end_idx]
            
            # Correlación con señal de referencia
            correlation = np.sum(bit_signal * ref_signal)
            
            detected_bits[i] = 1 if correlation > 0 else 0
        
        return detected_bits
    
    def calculate_ber(self, transmitted_bits: np.ndarray, received_bits: np.ndarray) -> float:
        """Calcular Bit Error Rate."""
        errors = np.sum(transmitted_bits != received_bits)
        return errors / len(transmitted_bits)

## Demostración: Formas de Onda sin Ruido

In [ ]:
# Parámetros de simulación
fc = 1000  # Frecuencia portadora (Hz)
fs = 8000  # Frecuencia de muestreo (Hz)
tb = 0.001  # Duración de bit (s)
A = 1.0    # Amplitud
N_demo = 8  # Número de bits para demostración

# Crear instancia del modulador
mod = DigitalModulation(fc, fs, tb, A)

# Generar bits de prueba
demo_bits = np.array([1, 0, 1, 1, 0, 1, 0, 0])
print(f"Bits de demostración: {demo_bits}")

# Generar señales moduladas
ask_signal = mod.ask_modulate(demo_bits)
fsk_signal = mod.fsk_modulate(demo_bits)
bpsk_signal = mod.bpsk_modulate(demo_bits)

# Vector de tiempo
t = np.linspace(0, len(demo_bits) * tb, len(ask_signal))

# Graficar formas de onda
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(t * 1000, ask_signal, 'b-', linewidth=1.5)
axes[0].set_title('ASK - Amplitude Shift Keying')
axes[0].set_ylabel('Amplitud')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t * 1000, fsk_signal, 'r-', linewidth=1.5)
axes[1].set_title('FSK - Frequency Shift Keying')
axes[1].set_ylabel('Amplitud')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t * 1000, bpsk_signal, 'g-', linewidth=1.5)
axes[2].set_title('BPSK - Binary Phase Shift Keying')
axes[2].set_ylabel('Amplitud')
axes[2].set_xlabel('Tiempo (ms)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Demostración: Señales con Ruido AWGN

In [ ]:
# Agregar ruido AWGN con SNR = 10 dB
snr_demo = 10  # dB

ask_noisy = mod.awgn_channel(ask_signal, snr_demo)
fsk_noisy = mod.awgn_channel(fsk_signal, snr_demo)
bpsk_noisy = mod.awgn_channel(bpsk_signal, snr_demo)

# Graficar señales con ruido
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(t * 1000, ask_noisy, 'b-', linewidth=1, alpha=0.8)
axes[0].set_title(f'ASK con ruido AWGN (SNR = {snr_demo} dB)')
axes[0].set_ylabel('Amplitud')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t * 1000, fsk_noisy, 'r-', linewidth=1, alpha=0.8)
axes[1].set_title(f'FSK con ruido AWGN (SNR = {snr_demo} dB)')
axes[1].set_ylabel('Amplitud')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t * 1000, bpsk_noisy, 'g-', linewidth=1, alpha=0.8)
axes[2].set_title(f'BPSK con ruido AWGN (SNR = {snr_demo} dB)')
axes[2].set_ylabel('Amplitud')
axes[2].set_xlabel('Tiempo (ms)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Constelación BPSK con SNR = 10 dB

In [ ]:
# Generar datos para constelación BPSK
N_constellation = 1000
constellation_bits = mod.generate_bits(N_constellation)
bpsk_constellation = mod.bpsk_modulate(constellation_bits)
bpsk_constellation_noisy = mod.awgn_channel(bpsk_constellation, 10)  # SNR = 10 dB

# Demodulación coherente para obtener puntos de constelación
I_samples = []
Q_samples = []

ref_cos = np.cos(2 * np.pi * fc * mod.t_bit)
ref_sin = -np.sin(2 * np.pi * fc * mod.t_bit)  # Componente en cuadratura

for i in range(N_constellation):
    start_idx = i * mod.samples_per_bit
    end_idx = (i + 1) * mod.samples_per_bit
    bit_signal = bpsk_constellation_noisy[start_idx:end_idx]
    
    # Correlación con portadoras en fase y cuadratura
    I = np.sum(bit_signal * ref_cos) / mod.samples_per_bit
    Q = np.sum(bit_signal * ref_sin) / mod.samples_per_bit
    
    I_samples.append(I)
    Q_samples.append(Q)

I_samples = np.array(I_samples)
Q_samples = np.array(Q_samples)

# Separar puntos por bit transmitido
I_bit0 = I_samples[constellation_bits == 0]
Q_bit0 = Q_samples[constellation_bits == 0]
I_bit1 = I_samples[constellation_bits == 1]
Q_bit1 = Q_samples[constellation_bits == 1]

# Graficar constelación
plt.figure(figsize=(10, 8))
plt.scatter(I_bit0, Q_bit0, c='red', marker='o', alpha=0.6, s=20, label='Bit 0')
plt.scatter(I_bit1, Q_bit1, c='blue', marker='s', alpha=0.6, s=20, label='Bit 1')

# Puntos ideales de constelación
plt.scatter([-A/2], [0], c='red', marker='x', s=200, linewidth=3, label='Ideal Bit 0')
plt.scatter([A/2], [0], c='blue', marker='x', s=200, linewidth=3, label='Ideal Bit 1')

plt.xlabel('I (En fase)')
plt.ylabel('Q (Cuadratura)')
plt.title(f'Constelación BPSK (SNR = 10 dB, N = {N_constellation} bits)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

print(f"Puntos de constelación generados: {len(I_samples)}")
print(f"Distribución de bits - 0: {np.sum(constellation_bits == 0)}, 1: {np.sum(constellation_bits == 1)}")

## Análisis BER vs SNR

In [ ]:
def calculate_ber_vs_snr(scheme: str, snr_values: List[float], N_bits: int = 10000) -> List[float]:
    """
    Calcular BER para diferentes valores de SNR.
    
    Args:
        scheme: 'ask', 'fsk', o 'bpsk'
        snr_values: Lista de valores SNR en dB
        N_bits: Número de bits para simulación
    
    Returns:
        Lista de valores BER
    """
    ber_values = []
    
    for snr_db in snr_values:
        # Generar bits aleatorios
        bits = mod.generate_bits(N_bits)
        
        # Modular según el esquema
        if scheme == 'ask':
            modulated_signal = mod.ask_modulate(bits)
        elif scheme == 'fsk':
            modulated_signal = mod.fsk_modulate(bits)
        elif scheme == 'bpsk':
            modulated_signal = mod.bpsk_modulate(bits)
        else:
            raise ValueError("Esquema debe ser 'ask', 'fsk' o 'bpsk'")
        
        # Pasar por canal AWGN
        received_signal = mod.awgn_channel(modulated_signal, snr_db)
        
        # Demodular
        if scheme == 'ask':
            detected_bits = mod.ask_demodulate(received_signal, N_bits)
        elif scheme == 'fsk':
            detected_bits = mod.fsk_demodulate(received_signal, N_bits)
        elif scheme == 'bpsk':
            detected_bits = mod.bpsk_demodulate(received_signal, N_bits)
        
        # Calcular BER
        ber = mod.calculate_ber(bits, detected_bits)
        ber_values.append(ber)
        
        print(f"Esquema: {scheme.upper()}, SNR: {snr_db:2d} dB, BER: {ber:.6f}")
    
    return ber_values

# Valores de SNR para análisis
snr_range = [20, 15, 10, 5, 0, -5]  # dB
N_analysis = 10000  # Número de bits para análisis

print("=== Análisis BER vs SNR ===")
print(f"Número de bits por simulación: {N_analysis}")
print("\n--- ASK ---")
ber_ask = calculate_ber_vs_snr('ask', snr_range, N_analysis)

print("\n--- FSK ---")
ber_fsk = calculate_ber_vs_snr('fsk', snr_range, N_analysis)

print("\n--- BPSK ---")
ber_bpsk = calculate_ber_vs_snr('bpsk', snr_range, N_analysis)

## Tabla de Resultados BER vs SNR

In [ ]:
# Crear DataFrame con resultados
results_df = pd.DataFrame({
    'SNR_dB': snr_range,
    'BER_ASK': ber_ask,
    'BER_FSK': ber_fsk,
    'BER_BPSK': ber_bpsk
})

print("\n=== TABLA DE RESULTADOS BER vs SNR ===")
print(results_df.to_string(index=False, float_format='%.6f'))

# Graficar curvas BER
plt.figure(figsize=(12, 8))
plt.semilogy(snr_range, ber_ask, 'bo-', linewidth=2, markersize=8, label='ASK')
plt.semilogy(snr_range, ber_fsk, 'rs-', linewidth=2, markersize=8, label='FSK')
plt.semilogy(snr_range, ber_bpsk, 'g^-', linewidth=2, markersize=8, label='BPSK')

plt.xlabel('SNR (dB)')
plt.ylabel('BER (Bit Error Rate)')
plt.title('Curvas BER vs SNR para Esquemas de Modulación Digital')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim([-6, 21])
plt.ylim([1e-5, 1])
plt.show()

# Exportar resultados a CSV
csv_filename = 'Semana2_BER_results.csv'
results_df.to_csv(csv_filename, index=False)
print(f"\n✓ Resultados exportados a: {csv_filename}")

## Análisis de Resultados

### Observaciones:

1. **BPSK**: Presenta el mejor rendimiento (menor BER) para todos los valores de SNR debido a su máxima separación entre símbolos en el espacio de señales.

2. **FSK**: Rendimiento intermedio. La separación entre frecuencias f₀ y f₁ afecta directamente el rendimiento.

3. **ASK**: Generalmente presenta mayor BER debido a su susceptibilidad a las variaciones de amplitud causadas por el ruido.

### Aplicaciones Prácticas:
- **BPSK**: Comunicaciones satelitales, sistemas de navegación GPS
- **FSK**: Módems de baja velocidad, transmisión de datos en HF
- **ASK**: Comunicaciones ópticas, algunos sistemas RFID

## Conclusiones

En esta práctica hemos implementado y analizado tres esquemas fundamentales de modulación digital:

1. **Implementación exitosa** de las ecuaciones matemáticas en código Python
2. **Visualización** de formas de onda y efectos del ruido AWGN
3. **Análisis cuantitativo** del rendimiento mediante curvas BER vs SNR
4. **Comparación** objetiva entre esquemas de modulación

Los resultados demuestran la importancia de la selección del esquema de modulación según los requerimientos específicos del sistema de comunicaciones.